In [20]:
import os
import torch
from PIL import Image
import numpy as np
from torchvision import transforms
from tqdm import tqdm

PATCH_SIZE = 32
STRIDE = 32
KANJI_DIR = "/app/data/jouyou"
SAVE_DIR = "/app/notebooks/kanji_features"
MODEL_PATH = "/app/notebooks/dino_radicals_resnet.pth"

os.makedirs(SAVE_DIR, exist_ok=True)

# Feature extractor
class DINOFeatureExtractor(torch.nn.Module):
    def __init__(self, model_path):
        super().__init__()
        from torchvision import models
        resnet = models.resnet18(pretrained=False)
        self.backbone = torch.nn.Sequential(*list(resnet.children())[:-1])
        self.projector = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(512, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 2048),
            torch.nn.GELU(),
            torch.nn.Linear(2048, 512),
        )
        self.load_state_dict(torch.load(model_path, map_location="cpu"))
        self.eval()

    def forward(self, x):
        x = self.backbone(x)
        x = self.projector(x)
        return x

# Extract non-overlapping patches
def extract_patches(img, patch_size=PATCH_SIZE, stride=STRIDE):
    w, h = img.size
    patches = []
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = img.crop((x, y, x + patch_size, y + patch_size))
            patches.append(patch)
    return patches

# Preprocess for DINO
def preprocess_patches(patches):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    return torch.stack([transform(p) for p in patches])

# Run
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DINOFeatureExtractor(MODEL_PATH).to(device)

for fname in tqdm(os.listdir(KANJI_DIR), desc="Extracting patch features"):
    if not fname.endswith(".png"):
        continue
    kanji_name = fname.replace(".png", "")
    img_path = os.path.join(KANJI_DIR, fname)
    img = Image.open(img_path).convert("L")
    patches = extract_patches(img)
    if len(patches) == 0:
        continue

    inputs = preprocess_patches(patches).to(device)
    with torch.no_grad():
        feats = model(inputs).cpu().numpy()  # shape: (num_patches, 512)

    out_path = os.path.join(SAVE_DIR, f"{kanji_name}.npy")
    np.save(out_path, feats)

print("✅ Patch features saved to:", SAVE_DIR)


Extracting patch features: 100%|██████████████████████████████████████████████████████████████| 2136/2136 [01:33<00:00, 22.73it/s]

✅ Patch features saved to: /app/notebooks/kanji_features


In [2]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import gc

# Parameters
radicals_feat_folder = "/app/notebooks/radical_features"
kanji_feat_folder = "/app/notebooks/kanji_features"
kanji_img_folder = "/app/data/jouyou"
output_dir = "/app/notebooks/kanji_visualizations"
PATCH_SIZE = 32
SIM_THRESHOLD = 0.90

os.makedirs(output_dir, exist_ok=True)
plt.ioff()  # Turn off interactive mode

# Load radical features
radical_features = {}
for fname in os.listdir(radicals_feat_folder):
    if fname.endswith(".npy"):
        radical_name = fname.replace(".npy", "")
        radical_features[radical_name] = np.load(os.path.join(radicals_feat_folder, fname))

radical_names = list(radical_features.keys())
radical_feats = np.stack([radical_features[name] for name in radical_names])  # (num_radicals, feat_dim)

# Loop over kanji features
for fname in tqdm(os.listdir(kanji_feat_folder), desc="Visualizing kanji"):
    if not fname.endswith(".npy"):
        continue

    kanji_name = fname.replace(".npy", "")
    kanji_feat = np.load(os.path.join(kanji_feat_folder, fname))  # shape (num_patches, feat_dim)

    if kanji_feat.ndim != 2:
        print(f"Skipping {kanji_name}, bad shape: {kanji_feat.shape}")
        continue

    # Compute cosine similarity (patch-by-radical)
    sims = cosine_similarity(kanji_feat, radical_feats)  # shape (num_patches, num_radicals)

    matches = []
    for i, sim_vec in enumerate(sims):
        top_rad_idx = np.argmax(sim_vec)
        top_sim = sim_vec[top_rad_idx]
        if top_sim >= SIM_THRESHOLD:
            x = (i % 7) * PATCH_SIZE
            y = (i // 7) * PATCH_SIZE
            matches.append({
                "patch_coord": (x, y),
                "radical": radical_names[top_rad_idx],
                "similarity": top_sim
            })

    if not matches:
        continue  # skip kanji with no matching patches

    # Load original image
    img_path = os.path.join(kanji_img_folder, f"{kanji_name}.png")
    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        continue
    img = Image.open(img_path).convert("L")

    # Draw
    fig, ax = plt.subplots()
    ax.imshow(img, cmap="gray")

    for m in matches[:5]:  # top 5 matches
        x, y = m["patch_coord"]
        radical = m["radical"]
        sim = m["similarity"]

        rect = patches.Rectangle((x, y), PATCH_SIZE, PATCH_SIZE, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 3, f"{radical} ({sim:.2f})", color='red', fontsize=6, backgroundcolor='white')

    ax.set_title(f"Kanji: {kanji_name}")
    ax.axis('off')

    out_path = os.path.join(output_dir, f"{kanji_name}.png")
    plt.savefig(out_path, dpi=120, bbox_inches='tight')  # reduced DPI
    fig.clf()
    plt.close(fig)

    del fig, ax, sims, matches, kanji_feat, img
    gc.collect()  # free memory


Visualizing kanji: 100%|██████████████████████████████████████████████████████████████████████| 2136/2136 [05:05<00:00,  6.98it/s]


In [22]:
from torchvision.utils import make_grid
import torch

# Load image as array (already grayscale)
img_np = np.array(img)  # shape: (224, 224)

# Extract top 5 patches as 32x32 crops
patch_imgs = []
for m in matches[:5]:
    x, y = m["patch_coord"]
    patch = img_np[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
    patch_tensor = torch.tensor(patch).unsqueeze(0) / 255.0  # shape: (1, 32, 32)
    patch_imgs.append(patch_tensor)

# Stack and save grid
if patch_imgs:
    patch_grid = make_grid(torch.stack(patch_imgs), nrow=5, padding=2)
    plt.figure(figsize=(5, 1.5))
    plt.imshow(patch_grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(f"Top Patches: {kanji_name}")
    plt.savefig(os.path.join(output_dir, f"{kanji_name}_patches.png"), bbox_inches="tight", dpi=120)
    plt.close()


NameError: name 'img' is not defined